# RecursiveGCD (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_java.ipynb)

Chapter 14 — Recursion. Runnable notebook of `RecursiveGCD`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/RecursiveGCD_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Greatest common divisor: RecursiveGCD

**Input:** two natural numbers $n \ge 1$ and $m \ge 0$. **Output:** $\gcd(n, m)$. Tail recursion: each call makes at most one, the tree is a path. At most $2\lfloor\log_2 m\rfloor + 3$ calls, $O(\log_2 m)$ steps.

Pseudo-code: section 14.6 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile RecursiveGCD.java
public class RecursiveGCD
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int m)                    // floor(log2 m) with a loop (only for the bound on the calls)
    {
        int k = 0;
        while (m > 1)
        {
            m = m / 2;
            k = k + 1;
        }
        return k;
    }

    // RecursiveGCD(n, m)
    static int recursiveGCD(int n, int m)
    {
        push(m); cnt[2]++;
        if (m == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("base case, returns %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("%d mod %d = %d: calls RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
        cnt[5]++;
        int g = recursiveGCD(m, n % m);
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("returns %d", g));
        pop();
        return g;
    }

    public static void main(String[] args)
    {
        // Examples of the lecture notes: RecursiveGCD(48, 18) and RecursiveGCD(252, 105), trees of calls (paths)
        int[][] pairs = {{48, 18}, {252, 105}};
        for (int[] p : pairs)
        {
            reset();
            trace = true;
            System.out.println(String.format("%-28s| %s", "call", "action"));
            int g = recursiveGCD(p[0], p[1]);
            trace = false;
            System.out.printf("gcd(%d, %d) = %d, calls = %d, S = %d\n", p[0], p[1], g, calls, total());
            System.out.printf("\n");
        }

        // Number of calls (each call executes 2 lines, hence S = 2 calls) and bound 2 floor(log2 m) + 3
        System.out.printf("      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3\n");
        int[][] more = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
        for (int[] p : more)
        {
            reset();
            int g = recursiveGCD(p[0], p[1]);
            System.out.printf("%7d | %7d | %4d | %8d | %3d | %d\n", p[0], p[1], g, calls, total(), 2 * floorLog2(p[1]) + 3);
        }
    }
}

Writing RecursiveGCD.java


In [3]:
!javac RecursiveGCD.java && java RecursiveGCD

call                        | action
RecursiveGCD(48, 18)        | 48 mod 18 = 12: calls RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: calls RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: calls RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | base case, returns 6
    RecursiveGCD(12, 6)     | returns 6
  RecursiveGCD(18, 12)      | returns 6
RecursiveGCD(48, 18)        | returns 6
gcd(48, 18) = 6, calls = 4, S = 8

call                        | action
RecursiveGCD(252, 105)      | 252 mod 105 = 42: calls RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: calls RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(42, 21)    | returns 21
  RecursiveGCD(105, 42)     | returns 21
RecursiveGCD(252, 105)      | returns 21
gcd(252, 105) = 21, calls = 4, S = 8

      n |       m |  gcd |    calls |   S | 2 floor(log2 m) + 3
     48 |

**Try it yourself.** Change the two numbers; the worst case is two consecutive Fibonacci numbers.

In [4]:
%%writefile ProvaRecursiveGCD.java
public class ProvaRecursiveGCD
{
    public static void main(String[] args)
    {
        int n = 1071, m = 462;        // <-- change the two numbers
        RecursiveGCD.reset();
        RecursiveGCD.trace = true;
        System.out.println(String.format("%-28s| %s", "call", "action"));
        int g = RecursiveGCD.recursiveGCD(n, m);
        RecursiveGCD.trace = false;
        System.out.printf("gcd(%d, %d) = %d, calls = %d\n", n, m, g, RecursiveGCD.calls);
    }
}

Writing ProvaRecursiveGCD.java


In [5]:
!javac ProvaRecursiveGCD.java && java ProvaRecursiveGCD

call                        | action
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: calls RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: calls RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: calls RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | base case, returns 21
    RecursiveGCD(147, 21)   | returns 21
  RecursiveGCD(462, 147)    | returns 21
RecursiveGCD(1071, 462)     | returns 21
gcd(1071, 462) = 21, calls = 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).